# 32 - Predictive Modeler
Similarly to `31_cleaning_tool.ipynb`, this notebook reflects the logic of another Python file, `22-interactive_predictive_modeler.py`. This notebook will use the clean version of the Titanic dataset to predict survival odds based on multiple other contributing variables using logistic regression.

It's important to note that the Python file itself serves as a walkthrough and provides instructions if the end user prompts the engine to "hold their hand." In this example, I will be opting out of the "hand holding" when running, specifically, but will still show the logic used to implement the set of instructions.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

### Introductory Output
The introductory output section will collect information on whether the end user would like to have instructions provided, then ask for the file path to whatever CSV the end user would like to use for predictive modeling.

In [5]:
print("\n")
print("Predictive Modeler here.")
print("If you would like me to hold you hand through this,")
print("type 'Y'")
print("Otherwise, skip or type anything else")

hold_hand = input("")




Predictive Modeler here.
If you would like me to hold you hand through this,
type 'Y'
Otherwise, skip or type anything else


 N


In [6]:
if hold_hand == "Y":
    print("No problem.")
    print("\n")
    print("---INSTRUCTIONS---")
    print("\n")
    print("In your files, find your csv table, pivot, or dataframe.")
    print("\n")
    print("Right click, and copy that table's file path.")
    print("\n")
    print("You will want to paste that file path below, but be sure")
    print("to remove the quotation marks surrounding the path.")
    print("\n")
else:
    print("File path (no quotes):")

File path (no quotes):


An important note, regardless, is that quotation marks should be removed from the path.

### Data Collection
The data collection section will provide an input variable for the CSV path, repeat back the end user what is being loaded, and ask that the user provides a name for their dataframe.

In [ ]:
data_input = str(input(""))

print("\n")
print("Excellent. We'll load up: ")
print("\n")
print(data_input)

print("\n")
if hold_hand == "Y":
    print("Check that. If it looks good, go ahead and give your")
    print("new dataframe a name.")
    print("\n")
else:
    print("Provide a name for your dataframe:")

data_name = input("")

# (I did punch in my file path but chose to leave it off the final copy...
# ...of the notebook for my own privacy -- named it "Titanic")

After collecting the data and its name, the Python script will load the data, convert it to a dataframe `df`, and offer a summary of the data.

In [8]:
print("\n")
print("Loading", data_name, "...")
data = pd.read_csv(data_input)
df = pd.DataFrame(data)
print("\n")
print("Success.")
print("\n")

print("\n")
enter_to_continue = input("Enter to continue")
print("\n")

print("Here is your summary.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isna().sum())

print("\n")
enter_to_continue = input("Enter to continue")
print("\n")



Loading Titanic ...


Success.






Enter to continue 




Here is your summary.
Shape: (891, 12)

Columns:
passengerid      int64
survived         int64
pclass           int64
name               str
sex                str
age            float64
sibsp            int64
parch            int64
ticket             str
fare           float64
cabin              str
embarked           str
dtype: object

Missing Values:
passengerid      0
survived         0
pclass           0
name             0
sex              0
age            177
sibsp            0
parch            0
ticket           0
fare             0
cabin          687
embarked         2
dtype: int64




Enter to continue 


The `enter_to_continue` is merely an input variable that allows for better compartmentalization of the outputs so that things don't get too clustered.

### The 'y' Column
The y-column is what we'll be predicting. For a sales pipeline, it might be win/loss. For sports, video games, etc., the y-column might be win/loss/draw. For the Titanic dataset, it'll be survives/dies. We will prompt the end user to enter the name of the column they'd like to predict, and then we will encode the possible values in the y-column. 

In [9]:
# First, we must instruct the user to enter their y-column's name.
if hold_hand == "Y":
    print("See the 'Columns:' section? Take a look at your column types.")
    print("'int64', 'float64', 'str', they're all different types.")
    print("'int64' refers to an integer number.")
    print("'float64' refers to a number who can have a decimal, followed by many places.")
    print("'str' is a string. That's text.")
    print("You may see any, all, or none of these particular types in YOUR data.")
    print("\n")
    print("For any sort predictive modeling, you must know WHICH variable you want to")
    print("predict. For instance, probability of winning/closing deal,")
    print("probability of repeat purchase, etc.")
    print("Make sure you know what you're predicting. This calculator is solely")
    print("for predicting success chances. So your y-column cannot be numeric")
    print("unless the numbers represent success/failure.")
    print("Think 'Win/Loss/Draw'. Think '0/1/2'. Not '16241, 27112, 62142, 72421', etc.")
    print("\n")

if hold_hand == "Y":
    print("Which COLUMN name are we predicting? Type it exactly as is.")
else:
    print("Title of column to predict:")

Title of column to predict:


In [10]:
# Here's the actual input:
y_column = input("")

 survived


In [11]:
print("\n")
print(df[y_column].unique()) # Prints the unique y-column values (i.e, ['Win', 'Loss'])
print("\n")

# Sees how many possible outcomes the y-column allows for
predictable_possibilities = len(df[y_column].unique())

if hold_hand == "Y":
    print("Looks like we have", predictable_possibilities, "possibilities.")
    print("We want to encode them.")

# Makes a list out of the unique possible outcomes
poss_array = df[y_column].unique().tolist()

# Encodes that list with index values / numbers
# Ex. Win/Loss/Draw = 0/1/2
encoded_poss = []
index = 0
for poss in poss_array:
    encoded_poss.append(index)
    index += 1

print(poss_array)

print(encoded_poss)
print("The above encoded possibilities list is what we will use as our possible,")
print("predictable outcomes.")
print("\n")
# Returns the encoded y-column possibilities to the end user.



[0 1]


[0, 1]
[0, 1]
The above encoded possibilities list is what we will use as our possible,
predictable outcomes.




In [12]:
# Now we want to zip the encoded possibilities back into the non-encoded possibilites.
# In other words, where the y-column says 'Win,' we want it to say '0,' or '1.'
mapping_dict = dict(zip(poss_array, encoded_poss))
df[y_column] = df[y_column].map(mapping_dict)

print(df[y_column].value_counts())

survived
0    549
1    342
Name: count, dtype: int64


### The 'X' Column(s)
The X-columns are incredibly important to the multivariate analysis because they are effectively what will be DOING the predicting. For instance, with a sales pipeline, there are multiple variables that impact the likelihood of a deal landing. The particular sales rep, the value of the deal, the maturity of the sales process, and the client's company all might correlate with the probability of a deal winning. In the X-column(s) section, we are going to prompt our end user to enter the columns they'd like to use to predict their outcome. For our Titanic example, these will be Age, Sex, Embarked, etc. 

In [13]:
# First, we will offer instructions to those who prompted the engine for instructions:
if hold_hand == "Y":
    print("In essence, the variable ", y_column, "will be our Y-axis.")
    print("In turn, we will model for X, and Y, where X predicts Y.")
    print("However, we will run a multivariate analysis using Logistic Regression,")
    print("meaning that we can have multiple X's.")
    print("In this next section, you will enter your X variables, or in")
    print("other words, the variables that you suspect may influence Y.")

In [14]:
# Now, we'll build a list of X-columns to be used in predictive modeling.
x_list = [] # Declare x_list as a list.
x_flag = True # Set a random variable flag as True.
print("Provide the titles of predictor (X) columns, one at a time, below,")
print("and simply type 'Done' when finished.")
while x_flag == True: # WHILE that variable flag is True, we'll do this:
    x_list_input = input("") # Give them a space to type their X-column.
    if x_list_input == 'Done': # If they type done,
        print("Just to clarify, the column names you entered are as follows:")
        for x in x_list:
            print("-", x) # We'll print out their list and make sure they like what they see.
        print("Correct? Y/N")
        correct_or_no = input("")
        if correct_or_no == "Y": # If they say, 'Y,' we're good,
            x_flag = False # We'll set the flag to false and kill the loop.
        else: # If they DON'T like what they see,
            x_list.clear() # We clear the list,
            print("Start from the top:") # And start over.
    else: # If they don't say 'Done,' and actually name an X-column,
        x_list.append(x_list_input) # We append the list with that X-column.

Provide the titles of predictor (X) columns, one at a time, below,
and simply type 'Done' when finished.


 pclass
 sex
 age
 embarked
 Done


Just to clarify, the column names you entered are as follows:
- pclass
- sex
- age
- embarked
Correct? Y/N


 Y


This covers the entirety of the X-column / list looping logic. Feel free to play with it in `22-interactive_predictive_modeler.py`.

In [12]:
# Lastly, we'll tell the end user we're going to build their engine.
print("Perfect.")
print("Let's build your engine.")

print("\n")
enter_to_continue = input("Enter to continue")
print("\n")

Perfect.
Let's build your engine.




Enter to continue 


### Dataframe Logic
Here's a problem. The engine will return an error if it tries to use ONLY the columns the user entered and not all the other columns in the dataframe. What we need to do is make a NEW dataframe that effectively drops the columns the user opted not to use in multivariate regression analysis. Luckily, this is easy.

In [15]:
# We'll make a list called dropcol (drop columns)
dropcol = []
# Then loop for columns in our dataframe that are NOT in x_list from earlier.
for col in df:
    if col not in x_list:
        dropcol.append(col) # And append those columns to dropcol.

# But we don't want the y_column dropped (even though it's not in x_list)
dropcol.remove(y_column)
# So we'll remove the y_column from dropcol.

# Now let's copy our dataframe
df1 = df
# And drop dropcol from it.
df1 = df1.drop(columns = dropcol)
# Easy does it.

### Building the Engine
The engine is easy enough to build with scikit-learn's prose. We'll need LogisticRegression, OneHotEncoder, ColumnTransformer, and Pipeline. The logic is as follows:

In [16]:
# The logistic regression model requires an X and a y.
X = df1[x_list] # X will be our x_list columns
y = df1[y_column] # y will be our y column.

In [18]:
# We put any numeric column into numeric_feats - a crucial step for the ColumnTransformer.
numeric_feats = df1.select_dtypes(include='number').columns.tolist()
if y_column in numeric_feats:
    numeric_feats.remove(y_column)

# Have to do something about NaNs if there are any:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# We put any NON-numeric column into categorical_feats.
categorical_feats = df1.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

# Have to do something about NaNs if there are any:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


In [19]:
# Now we'll define "preprocess," a critical variable to the model-building step that comes next.
preprocess = ColumnTransformer(
    transformers = [
        ("categories", categorical_transformer, categorical_feats),
        ("numeric", numeric_transformer, numeric_feats)
    ]
)

In [20]:
# And we'll build the model:
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(max_iter = 9999))  
])
# Note: max_iter = 9999 may need to be changed.
# I'm not entirely sure where more/less iterations would be needed,
# But I do know I had too little at one point and the engine raised an error.
# For most datasets you can find online, 9999 should be fine.

In [21]:
# And we'll jam our X and y into the model:
model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('logreg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categories', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [22]:
# Lastly, we'll show our end user what we just did:
print("\n")
print("X = df[x_list]")
print("y = df[y_column]")
print("numeric_feats = df.select_dtypes(include='number').columns.tolist()")
print("categorical_feats = df.select_dtypes(include=['object', 'category']).columns.tolist()")
print("Numeric Features:", numeric_feats)
print("Categorical Features:", categorical_feats)
print("preprocess = ColumnTransformer(")
print("    transformers = [")
print("        ('categories', OneHotEncoder(drop = 'first'), categorical_feats),")
print("        ('numeric', 'passthrough', numeric_feats)")
print("    ]")
print(")")
print("model = Pipeline(steps=[")
print("    ('preprocess', preprocess),")
print("    ('logreg', LogisticRegression(max_iter = 9999))")
print("])")
print("model.fit(X, y)")
print("\n")

print("Engine was successfully created.")

print("\n")
print("\n")
# We just print exactly what we typed before. Looks cool.
# Makes the engine look as if it's being constructed right before the user's eyes.



X = df[x_list]
y = df[y_column]
numeric_feats = df.select_dtypes(include='number').columns.tolist()
categorical_feats = df.select_dtypes(include=['object', 'category']).columns.tolist()
Numeric Features: ['pclass', 'age']
Categorical Features: ['sex', 'embarked']
preprocess = ColumnTransformer(
    transformers = [
        ('categories', OneHotEncoder(drop = 'first'), categorical_feats),
        ('numeric', 'passthrough', numeric_feats)
    ]
)
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('logreg', LogisticRegression(max_iter = 9999))
])
model.fit(X, y)


Engine was successfully created.






### Calculator
The calculator is where the fun begins for the end user. Now that they've set up their engine, they get to play around with it. They'll be prompted to enter in their own scenarios, and the engine will return the likelihood of each of their y_column outputs happening based on the logarithms.

In [23]:
# First, we can tell the user what's going on:
if hold_hand == "Y":
    print("\n")
    enter_to_continue = input("Enter to continue")
    print("\n")
    print("Now you will get to tinker with your engine.")
    print("You will be prompted to enter values for your various")
    print("X columns, and when you've done so, it will return the")
    print("success probability in the same order it was in when")
    print("you were shown the y-column possible outcomes from earlier.")
    print("Feel free to play around as much or as little as you'd like.")
    print("When you're all done, either kill the program or type N to")
    print("break out of the loop and kill the program automatically.")
    print("Next time, you can elect for me NOT to hold your hand")
    print("throughout if you'd prefer, and the process will be more")
    print("streamlined.")
    print("\n")
    print("Enjoy!")
    print("\n")

print("\n")
enter_to_continue = input("Enter to continue")
print("\n")

Enter to continue 


In [31]:
# Now for the calculator:
flag = False # Set a flag to False
while flag == False: # While that flag is false,
    print("---Calculator Input---")
    predictor_dict = {} # We establish an empty (for now) dictonary.
    for x in x_list: # For each column they gave us earlier,
        print("Enter", x, "below") # We'll have them enter a value...
        test = input("") # Right here.
        x_value = str(x) 
        if x in numeric_feats: # But IF it's a numeric column,
            predictor_dict[x_value] = [float(test)] # We need to float the value.
        else: # Otherwise, we'll set their value as a string.
            predictor_dict[x_value] = [test]
    predictor = pd.DataFrame(predictor_dict) # Whether obvious or not, we now have a dataframe.
    prediction = model.predict_proba(predictor) # predict_proba predicts the likelihood of the y-Column possibilites
    print("---Calculator Output---")
    print(prediction) # Output the prediction
    print("\n")
    again = input("Again? Y/N") # Ask them if they want to go again with different values.
    if again == "N": # If no, the Python file is killed
        flag = True
        exit
    else: # Otherwise, the flag remains False and the loop entirely restarts.
        flag = False

---Calculator Input---
Enter pclass below


 1


Enter sex below


 female


Enter age below


...

 60


Enter embarked below


 S


---Calculator Output---
[[0.96962564 0.03037436]]




Again? Y/N N


For the Titanic example, here, we see that women/children/1st-class passengers yielded the highest probability of survival. Furthermore, from where one departed tended to correlate ever-so-slightly, where England (S) departures unfortunately yielded a slightly lesser chance of survival than France/Ireland (Q/C). The calculator will let you keep running and running however many times you'd like to uncover whatever insight you're interested in uncovering.